In [ ]:
"""
ICON-NET: Integrated Channel-Spatial and Gate Attention Network
for Brain Tumor Segmentation Across Multi-Modal MRI Datasets

Architecture:
    - 4-level UNet encoder-decoder
    - CBAM (channel + spatial attention) in every ConvBlock
    - AttentionGate on every decoder skip connection
    - Deep supervision at 3 decoder levels (d4, d3, d2)
    - Residual connections in every ConvBlock

Reference:
    [Paper title and citation to be added upon publication]

Usage:
    from iconnet import ICONNET
    model = ICONNET(ic=3, oc=1, base=64, dr=0.3)
    model.train()
    out, ds4, ds3, ds2 = model(x)   # training mode — returns 4 outputs
    model.eval()
    out = model(x)                  # inference mode — returns main output only
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Attention Modules ────────────────────────────────────────

class ChannelAttention(nn.Module):
    """
    CBAM Channel Attention.
    Recalibrates channel-wise feature responses using
    global average pooling and global max pooling.

    Args:
        c (int): Number of input channels.
        r (int): Reduction ratio for the MLP bottleneck. Default: 16.
    """
    def __init__(self, c: int, r: int = 16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(
            nn.Conv2d(c, c // r, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(c // r, c, 1, bias=False),
        )
        self.sig = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))


class SpatialAttention(nn.Module):
    """
    CBAM Spatial Attention.
    Highlights spatially informative regions using
    channel-wise average and max projections.

    Args:
        k (int): Convolution kernel size. Default: 7.
    """
    def __init__(self, k: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)
        self.sig  = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1, keepdim=True)[0]
        return x * self.sig(self.conv(torch.cat([avg, mx], dim=1)))


class CBAM(nn.Module):
    """
    Convolutional Block Attention Module (CBAM).
    Applies channel attention followed by spatial attention sequentially.

    Reference:
        Woo et al., CBAM: Convolutional Block Attention Module, ECCV 2018.

    Args:
        c (int): Number of input channels.
        r (int): Channel attention reduction ratio. Default: 16.
    """
    def __init__(self, c: int, r: int = 16):
        super().__init__()
        self.ca = ChannelAttention(c, r)
        self.sa = SpatialAttention()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.sa(self.ca(x))


class AttentionGate(nn.Module):
    """
    Attention Gate for decoder skip connections.
    Selectively suppresses task-irrelevant encoder features
    using a gating signal from the deeper decoder stage.

    Reference:
        Oktay et al., Attention U-Net, MIDL 2018.

    Args:
        Fg (int): Channels in gating signal g (from decoder).
        Fl (int): Channels in skip feature x_l (from encoder).
        Fi (int): Intermediate channel dimension (bottleneck).
    """
    def __init__(self, Fg: int, Fl: int, Fi: int):
        super().__init__()
        self.Wg   = nn.Sequential(nn.Conv2d(Fg, Fi, 1, bias=True), nn.BatchNorm2d(Fi))
        self.Wx   = nn.Sequential(nn.Conv2d(Fl, Fi, 1, bias=True), nn.BatchNorm2d(Fi))
        self.psi  = nn.Sequential(nn.Conv2d(Fi, 1,  1, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        g_up = F.interpolate(self.Wg(g), size=x.shape[2:], mode='bilinear', align_corners=False)
        return x * self.psi(self.relu(g_up + self.Wx(x)))


# ── Core Building Block ──────────────────────────────────────

class ConvBlock(nn.Module):
    """
    Enhanced Convolutional Block with CBAM and residual connection.

    Structure:
        Conv3x3 → BN → ReLU → Dropout2d
        Conv3x3 → BN → ReLU → Dropout2d
        CBAM (channel + spatial attention)
        + Residual shortcut (1x1 conv if channel mismatch, else identity)

    Args:
        ic  (int):   Input channels.
        oc  (int):   Output channels.
        dr  (float): Dropout2d probability. Default: 0.3.
    """
    def __init__(self, ic: int, oc: int, dr: float = 0.3):
        super().__init__()
        self.c1   = nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1, bias=False),
            nn.BatchNorm2d(oc), nn.ReLU(inplace=True), nn.Dropout2d(dr),
        )
        self.c2   = nn.Sequential(
            nn.Conv2d(oc, oc, 3, padding=1, bias=False),
            nn.BatchNorm2d(oc), nn.ReLU(inplace=True), nn.Dropout2d(dr),
        )
        self.cbam = CBAM(oc)
        self.skip = (
            nn.Sequential(nn.Conv2d(ic, oc, 1, bias=False), nn.BatchNorm2d(oc))
            if ic != oc else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.cbam(self.c2(self.c1(x))) + self.skip(x)


# ── Loss Function ────────────────────────────────────────────

class CombinedLoss(nn.Module):
    """
    Combined Dice + BCE + Focal loss for binary segmentation.

    L = dw * L_Dice + bw * L_BCE + fw * L_Focal

    Args:
        dw     (float): Dice loss weight.   Default: 0.5.
        bw     (float): BCE loss weight.    Default: 0.3.
        fw     (float): Focal loss weight.  Default: 0.2.
        smooth (float): Dice smoothing constant. Default: 1e-6.
    """
    def __init__(self, dw: float = 0.5, bw: float = 0.3, fw: float = 0.2, smooth: float = 1e-6):
        super().__init__()
        self.dw = dw
        self.bw = bw
        self.fw = fw
        self.s  = smooth

    def dice(self, p: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        pf, tf = p.view(-1), t.view(-1)
        return 1 - (2 * (pf * tf).sum() + self.s) / (pf.sum() + tf.sum() + self.s)

    def focal(self, p: torch.Tensor, t: torch.Tensor, a: float = 0.25, g: float = 2.0) -> torch.Tensor:
        bce = F.binary_cross_entropy(p, t, reduction='none')
        return (a * (1 - torch.exp(-bce)) ** g * bce).mean()

    def forward(self, p: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy(p, t)
        return self.dw * self.dice(p, t) + self.bw * bce + self.fw * self.focal(p, t)


# ── ICON-NET ─────────────────────────────────────────────────

class ICONNET(nn.Module):
    """
    ICON-NET: Integrated Channel-Spatial and Gate Attention Network.

    A 4-level UNet encoder-decoder with:
        - CBAM attention in every ConvBlock (encoder + decoder)
        - AttentionGate on every decoder skip connection
        - Deep supervision at decoder levels d4, d3, d2 (training only)
        - Residual shortcuts in every ConvBlock

    Args:
        ic   (int):   Input channels. Default: 3 (RGB MRI slice).
        oc   (int):   Output channels. Default: 1 (binary mask).
        base (int):   Base channel width. Default: 64.
        dr   (float): Dropout2d rate. Default: 0.3.

    Encoder channel progression:
        Input → base → base*2 → base*4 → base*8 → base*16 (bottleneck)

    Returns (training):
        main (Tensor): [B, 1, H, W] — primary segmentation output
        ds4  (Tensor): [B, 1, H, W] — deep supervision from d4
        ds3  (Tensor): [B, 1, H, W] — deep supervision from d3
        ds2  (Tensor): [B, 1, H, W] — deep supervision from d2

    Returns (eval):
        main (Tensor): [B, 1, H, W] — primary segmentation output only

    Example:
        >>> model = ICONNET(ic=3, oc=1, base=64, dr=0.3).cuda()
        >>> x = torch.randn(2, 3, 256, 256).cuda()
        >>> model.train(); out = model(x)   # returns (main, ds4, ds3, ds2)
        >>> model.eval();  out = model(x)   # returns main only
    """
    def __init__(self, ic: int = 3, oc: int = 1, base: int = 64, dr: float = 0.3):
        super().__init__()
        b = base

        # Initial projection
        self.init_conv = nn.Sequential(
            nn.Conv2d(ic, b, 3, padding=1, bias=False),
            nn.BatchNorm2d(b),
            nn.ReLU(inplace=True),
        )

        # Encoder
        self.e1 = ConvBlock(b,    b,    dr);  self.p1 = nn.MaxPool2d(2)
        self.e2 = ConvBlock(b,    b*2,  dr);  self.p2 = nn.MaxPool2d(2)
        self.e3 = ConvBlock(b*2,  b*4,  dr);  self.p3 = nn.MaxPool2d(2)
        self.e4 = ConvBlock(b*4,  b*8,  dr);  self.p4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = ConvBlock(b*8, b*16, dr)

        # Attention gates
        self.ag4 = AttentionGate(b*16, b*8,  b*4)
        self.ag3 = AttentionGate(b*8,  b*4,  b*2)
        self.ag2 = AttentionGate(b*4,  b*2,  b)
        self.ag1 = AttentionGate(b*2,  b,    b//2)

        # Decoder
        self.up4 = nn.ConvTranspose2d(b*16, b*8, 2, stride=2)
        self.d4  = ConvBlock(b*16, b*8,  dr)

        self.up3 = nn.ConvTranspose2d(b*8,  b*4, 2, stride=2)
        self.d3  = ConvBlock(b*8,  b*4,  dr)

        self.up2 = nn.ConvTranspose2d(b*4,  b*2, 2, stride=2)
        self.d2  = ConvBlock(b*4,  b*2,  dr)

        self.up1 = nn.ConvTranspose2d(b*2,  b,   2, stride=2)
        self.d1  = ConvBlock(b*2,  b,    dr)

        # Output head
        self.out_conv = nn.Conv2d(b, oc, 1)

        # Deep supervision heads (training only)
        self.ds4 = nn.Conv2d(b*8,  oc, 1)
        self.ds3 = nn.Conv2d(b*4,  oc, 1)
        self.ds2 = nn.Conv2d(b*2,  oc, 1)

    def forward(self, x: torch.Tensor):
        # Initial projection
        x0 = self.init_conv(x)

        # Encoder
        e1 = self.e1(x0)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        e4 = self.e4(self.p3(e3))

        # Bottleneck
        bt = self.bottleneck(self.p4(e4))

        # Decoder with attention gates
        d4 = self.d4(torch.cat([self.up4(bt), self.ag4(bt, e4)], dim=1))
        d3 = self.d3(torch.cat([self.up3(d4), self.ag3(d4, e3)], dim=1))
        d2 = self.d2(torch.cat([self.up2(d3), self.ag2(d3, e2)], dim=1))
        d1 = self.d1(torch.cat([self.up1(d2), self.ag1(d2, e1)], dim=1))

        # Main output
        main = torch.sigmoid(self.out_conv(d1))

        # Deep supervision (training only)
        if self.training:
            s   = x.shape[2:]
            ds4 = torch.sigmoid(F.interpolate(self.ds4(d4), size=s, mode='bilinear', align_corners=False))
            ds3 = torch.sigmoid(F.interpolate(self.ds3(d3), size=s, mode='bilinear', align_corners=False))
            ds2 = torch.sigmoid(F.interpolate(self.ds2(d2), size=s, mode='bilinear', align_corners=False))
            return main, ds4, ds3, ds2

        return main


# ── Utilities ────────────────────────────────────────────────

def count_params(model: nn.Module) -> tuple:
    """Returns (total_params, trainable_params)."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def build_model(ic: int = 3, oc: int = 1, base: int = 64, dr: float = 0.3) -> ICONNET:
    """Convenience factory function."""
    return ICONNET(ic=ic, oc=oc, base=base, dr=dr)